# 04 · RQ3: generalisation to unseen voice generators

Applies the RQ2 models (thresholds fixed on validation) to **Test-Unknown** (11 generation systems never seen in training), reports per-system detection rates, and compares supervised models with one-class detectors trained only on human speech.

H₀: detection rate on unseen systems equals that on known systems. H₁: it is lower.

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

In [ ]:
import joblib, torch
from src import evaluation as E, models as M
from src.features import FEATURES
sample = pd.read_csv(C.META_DIR / "split_assignments.csv")
X = pd.read_csv(C.FEAT_DIR / "features_clean.csv"); L = np.load(C.FEAT_DIR / "logmel_clean.npy")
df = sample.merge(X, on="clip_id"); y = df.label.values
idx = {s: np.where(df.split == s)[0] for s in df.split.unique()}
tr, va = idx["training"], idx["validation"]
tk = np.r_[idx["test_human"], idx["test_known"]]; tu = np.r_[idx["test_human"], idx["test_unknown"]]
state = pd.read_pickle(C.MODEL_DIR / "rq2_state.pkl")
ck = torch.load(C.MODEL_DIR / "CNN.pt", weights_only=False)
cnn = M.build_cnn(); cnn.load_state_dict(ck["state_dict"])
def score(name, rows):
    if name == "CNN":
        return M.predict_cnn(cnn, L[rows], ck["mean"], ck["std"])
    return joblib.load(C.MODEL_DIR / f"{name}.joblib")["model"].predict_proba(df.loc[rows, FEATURES])[:, 1]

## Known versus unseen systems

In [ ]:
rows = []
for name in ["LR", "SVM", "RF", "XGBoost", "CNN"]:
    thr = state["thresholds"][name]
    for label, r in [("known", tk), ("unseen", tu)]:
        rows.append({"model": name, "systems": label, **E.classification_report(y[r], score(name, r), thr)})
ku = pd.DataFrame(rows); ku.to_csv(C.REPORT_DIR / "rq3_known_vs_unseen.csv", index=False)
ku.pivot(index="model", columns="systems", values=["eer", "recall_tpr", "fpr"]).round(4)

## Per-system detection rate (best supervised model) with speaker-cluster bootstrap CIs

In [ ]:
best = min(["LR", "SVM", "RF", "XGBoost", "CNN"], key=lambda n: ku[(ku.model == n) & (ku.systems == "unseen")].eer.iloc[0])
thr = state["thresholds"][best]
syn = np.r_[idx["test_known"], idx["test_unknown"]]
t = df.loc[syn, ["speaker_id", "attack_system_id", "attack_type", "system_status"]].assign(detected=(score(best, syn) >= thr).astype(int))
per = []
for sys_id, g in t.groupby("attack_system_id"):
    est, lo, hi = E.cluster_bootstrap(g, lambda d: d.detected.mean(), n_boot=1000)
    per.append({"system": sys_id, "type": g.attack_type.iloc[0], "status": g.system_status.iloc[0], "n": len(g),
                "detection_rate": est, "ci_low": lo, "ci_high": hi})
per = pd.DataFrame(per).sort_values("detection_rate"); per.to_csv(C.REPORT_DIR / "rq3_per_system.csv", index=False)
print("Best supervised model on unseen systems:", best); per.round(3)

In [ ]:
per.groupby(["status", "type"]).detection_rate.mean().round(3)

## One-class detectors trained on human speech only

In [ ]:
human_tr = tr[y[tr] == 0]
oc_rows, oc_scores = [], {}
for kind in ["IsolationForest", "OneClassSVM"]:
    oc = M.fit_one_class(kind, df.loc[human_tr, FEATURES])
    thr_oc = E.eer(y[va], M.one_class_score(oc, df.loc[va, FEATURES]))[1]
    s = M.one_class_score(oc, df.loc[tu, FEATURES]); oc_scores[kind] = (s, thr_oc)
    oc_rows.append({"model": kind, **E.classification_report(y[tu], s, thr_oc)})
    joblib.dump({"model": oc, "threshold": thr_oc}, C.MODEL_DIR / f"{kind}.joblib")
oc_res = pd.DataFrame(oc_rows); oc_res.to_csv(C.REPORT_DIR / "rq3_one_class.csv", index=False)
best_oc = oc_res.sort_values("eer").model.iloc[0]
s_oc, thr_oc = oc_scores[best_oc]
mc = E.mcnemar((score(best, tu) >= thr).astype(int) == y[tu], (s_oc >= thr_oc).astype(int) == y[tu])
print(f"{best} vs {best_oc} on Test-Unknown: b={mc['b']} c={mc['c']} p={mc['p_value']:.4f} ({mc['method']})")
oc_res.round(4)